# segments_reduced 是什么

`integrate_fermi_surface_observables(model, segments_reduced, recip_lat_vecs)` 里的 `segments_reduced` 是费米面折线列表。

更具体地说：

- `segments_reduced` 是一个 `list[np.ndarray]`。
- 列表里的每个 `segment` 是一个形状为 `(N, 2)` 的数组。
- 每一行是一个 reduced reciprocal coordinate：`[k1, k2]`。
- 这些点连起来就是一条满足 `E_lower(k1, k2) = mu` 的等能线，也就是二维体系里的费米线。

所以 `segment` 不是能量，也不是速度；它只是费米线在 reduced k 坐标里的采样点。后面的积分会把它转成 Cartesian k 坐标，计算每一小段长度 `ds`。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd()
if repo.name == "learning_code":
    repo = repo.parents[1]
elif not (repo / "quantum_geometry").exists():
    repo = Path("/home/mayuan/code/sctheory")

tb_dir = repo / "quantum_geometry" / "graphene" / "tb"
sys.path.insert(0, str(tb_dir))

from core import (  # noqa: E402
    GrapheneParameters,
    build_model,
    uniform_mesh_quantities,
    extract_fermi_surface_segments,
    extract_dirac_pocket_segments,
    reduced_k_to_cartesian,
)

## 1. 建立石墨烯模型并计算均匀 k 网格

`uniform_mesh_quantities()` 会在 reduced k 坐标的 `[0, 1) x [0, 1)` 区域生成均匀网格，并计算下带能量、能隙、量子度规等。这里我们只用下带能量来抽取费米线。

In [ ]:
params = GrapheneParameters()
model = build_model(params)

mesh_size = 180
mesh = uniform_mesh_quantities(model, mesh_size=mesh_size)

lower_band = mesh["evals_ev"][:, 0]
recip_lat_vecs = mesh["recip_lat_vecs"]

print("k_pts shape:", mesh["k_pts"].shape)
print("lower_band shape:", lower_band.shape)
print("energy range:", lower_band.min(), lower_band.max())

## 2. 用 contour 抽取一条普通费米线

`extract_fermi_surface_segments(lower_band, mu, mesh_size)` 的本质是对二维函数 `E_lower(k1,k2)` 画等高线。

这里 `mu = -0.30 eV`，费米线比较大，适合用全局网格 contour 抽取。

In [ ]:
mu_global = -0.30
segments_global = extract_fermi_surface_segments(lower_band, mu_global, mesh_size)

print("number of global segments:", len(segments_global))
for i, seg in enumerate(segments_global):
    print(f"segment {i}: shape = {seg.shape}")
    print(seg[:5])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.scatter(mesh["k_pts"][:, 0], mesh["k_pts"][:, 1], s=1, c="0.88", alpha=0.5, label="uniform k mesh")

for i, seg in enumerate(segments_global):
    ax.plot(seg[:, 0], seg[:, 1], lw=2.2, label=f"segment {i}, {seg.shape[0]} points")
    ax.scatter(seg[0, 0], seg[0, 1], s=35, marker="o")

ax.scatter([2/3, 1/3], [1/3, 2/3], marker="x", s=80, c="black", label="K / K'")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("reduced k1")
ax.set_ylabel("reduced k2")
ax.set_title(f"segments_reduced from global contour, mu={mu_global} eV")
ax.legend(fontsize=8)
plt.show()

## 3. 在 Dirac 点附近构造局部 pocket

当 `mu` 很接近 Dirac 点时，费米线是 K 和 K' 附近两个很小的 pocket。粗网格 contour 容易抽坏，所以代码里用 `extract_dirac_pocket_segments()` 沿角度方向逐点求根。

这个函数返回的仍然是同一种东西：`list[np.ndarray]`，每个数组依然是一条 reduced k 坐标下的费米线折线。

In [ ]:
mu_dirac = -0.02
segments_dirac = extract_dirac_pocket_segments(model, recip_lat_vecs, mu_dirac, num_angles=360)

print("number of Dirac pocket segments:", len(segments_dirac))
for i, seg in enumerate(segments_dirac):
    print(f"segment {i}: shape = {seg.shape}")
    print(seg[:5])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.scatter(mesh["k_pts"][:, 0], mesh["k_pts"][:, 1], s=1, c="0.9", alpha=0.4, label="uniform k mesh")

for i, seg in enumerate(segments_dirac):
    ax.plot(seg[:, 0], seg[:, 1], lw=2.2, label=f"Dirac pocket {i}, {seg.shape[0]} points")
    ax.scatter(seg[0, 0], seg[0, 1], s=35, marker="o")

ax.scatter([2/3, 1/3], [1/3, 2/3], marker="x", s=90, c="black", label="K / K'")
ax.set_xlim(0.20, 0.80)
ax.set_ylim(0.20, 0.80)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("reduced k1")
ax.set_ylabel("reduced k2")
ax.set_title(f"segments_reduced from local Dirac pockets, mu={mu_dirac} eV")
ax.legend(fontsize=8)
plt.show()

## 4. 为什么积分函数需要这些 segment

`integrate_fermi_surface_observables()` 会对每个 `segment` 做下面几步：

1. 把 reduced 坐标转成 Cartesian reciprocal 坐标。
2. 用相邻点差分得到每一小段长度 `ds`。
3. 在每一小段中点重新计算能量、gap、metric trace 和速度。
4. 累加费米线积分：

\[
\oint \frac{ds}{|v|},\qquad
\oint ds\,|v|,\qquad
\oint ds\,\frac{\Delta E^2\,\mathrm{Tr}\,g}{|v|},\qquad
\oint ds\,\frac{|v|}{\Delta E^2}.
\]

所以 `segments_reduced` 的作用只是告诉积分函数：费米线在哪里。真正的物理量是在这些费米线点上重新采样得到的。

In [ ]:
seg = segments_dirac[0]
seg_cart = reduced_k_to_cartesian(seg, recip_lat_vecs)
ds = np.linalg.norm(seg_cart[1:] - seg_cart[:-1], axis=1)
mid_reduced = 0.5 * (seg[:-1] + seg[1:])

print("one segment in reduced coords:", seg.shape)
print("same segment in cartesian reciprocal coords:", seg_cart.shape)
print("ds shape:", ds.shape)
print("first 5 ds:", ds[:5])
print("midpoints shape:", mid_reduced.shape)